# Experiment Training Notebook

Cold-starts from pre-computed `.npz` + warm-state files produced by `combined_data_split.ipynb` (both depth and no-depth variants). Runs a focused sweep of `(dataset, feature_subset, architecture, loss, lr_strategy)` combinations, sharded across **4 VMs** (`MY_SHARD` selects this VM's slice). Each run appends one row to `experiment_log__shard{MY_SHARD}.csv`; the **Results viewer** at the bottom merges every `experiment_log__shard*.csv` it finds.

**Sweep design**: loss types were pre-selected per architecture via a prior sweep in `combined_data_split.ipynb`. This notebook only varies feature subsets and lr strategy (fixed vs lr_find) for the winning arch-loss pairs.


In [ ]:
import pandas as pd
import os
import socket
import subprocess
import uuid
import gc as _gc
import sys as _sys
from pathlib import Path
import numpy as np
import time
from datetime import datetime
import json as _json

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from fastai.learner import *
from fastai.metrics import accuracy
from fastai.callback.tracker import EarlyStoppingCallback, SaveModelCallback
from fastai.losses import CrossEntropyLossFlat, FocalLossFlat, LabelSmoothingCrossEntropyFlat
from IPython.display import clear_output, display

from geoxts.benchmark_data import *
import geoxts.benchmark_data as _bd
from geoxts.benchmark_data import Well_grlog_cleaning as _orig_Well_grlog_cleaning
from geoxts.benchmark_models import *
from tsai.all import TSDatasets, TSDataLoaders, TSStandardize, Categorize, combine_split_data
from tsai.all import FCN, LSTM, XCM, LSTM_FCN

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Identity for cross-machine logging
MACHINE = socket.gethostname()
try:
    GIT_SHA = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
    ).decode().strip()
except Exception:
    GIT_SHA = "unknown"
print(f"Machine: {MACHINE} | git: {GIT_SHA}")


In [ ]:
# Fix random seed AND enforce fully deterministic CUDA for reproducible training.
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'   # required by torch deterministic mode

random_seed(0, use_cuda=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# warn_only=False -- raises RuntimeError on any non-deterministic op.
torch.use_deterministic_algorithms(True)
print("Fully deterministic mode enabled")


## Cold load: per-dataset loaders (depth vs no-depth)

Each dataset variant produced by `combined_data_split.ipynb` lives in its own subdirectory:

| `dataset_tag` | Directory | Files |
|---|---|---|
| `"nodepth"` | `no_depth/` | `train.npz`, `val.npz`, `test.npz`, `warm_df_test_g.parquet`, `warm_tops.parquet`, `warm_state.json` |
| `"depth"` | `depth/` | `train_depth.npz`, `val_depth.npz`, `test_depth.npz`, `warm_df_test_g_depth.parquet`, `warm_tops_depth.parquet`, `warm_state_depth.json` |

`load_dataset(tag)` returns everything the loop needs and (re-)installs the depth normalization monkey-patch on `Well_grlog_cleaning` when the tag has `USE_DEPTH=True`.

In [ ]:
# ─── Dataset registry: tag -> (directory, file suffix) ─────────────────────
# Matches the directory layout and `_DEPTH_TAG` convention from combined_data_split.ipynb.
# no_depth/ files have no suffix; depth/ files keep the "_depth" suffix.
DATASET_PATHS = {
    "nodepth": {"dir": "no_depth", "suffix": ""},
    "depth":   {"dir": "depth",    "suffix": "_depth"},
}


def _file(dataset_tag, basename_template):
    """Build the full path for a dataset artifact.

    Example: _file("depth", "train{suffix}.npz") -> "depth/train_depth.npz"
    """
    info = DATASET_PATHS[dataset_tag]
    return str(Path(info["dir"]) / basename_template.format(suffix=info["suffix"]))


def _install_depth_patch(depth_bounds):
    """Re-apply the depth normalization monkey-patch on Well_grlog_cleaning.

    The training-time pipeline normalizes Depth in [0,1] using min-max bounds
    via a monkey-patch in combined_data_split.ipynb cell 34. The patch is not
    saved to the .npz files, so prediction-time `get_markers` re-runs the raw
    `Well_grlog_cleaning` and would feed unnormalized depth to the model
    unless we re-install the patch here.
    """
    def _wgc_with_depth_norm(well_name, df_log, input_variable):
        df_wgr = _orig_Well_grlog_cleaning(well_name, df_log, input_variable)
        if 'Depth' in df_wgr.columns:
            lo, hi = depth_bounds
            df_wgr['Depth'] = ((df_wgr['Depth'] - lo) / (hi - lo)).clip(0, 1)
        return df_wgr
    _wgc_with_depth_norm._depth_patched = True
    _bd.Well_grlog_cleaning = _wgc_with_depth_norm
    globals()['Well_grlog_cleaning'] = _wgc_with_depth_norm


def _uninstall_depth_patch():
    """Restore the original (un-patched) Well_grlog_cleaning."""
    _bd.Well_grlog_cleaning = _orig_Well_grlog_cleaning
    globals()['Well_grlog_cleaning'] = _orig_Well_grlog_cleaning


def load_dataset(dataset_tag):
    """Cold-load all artifacts for a dataset variant.

    Returns a dict with X_train/y_train/X_val/y_val/X_test/y_test, df_test_g,
    tops, test_wells, well2id, MARKERS, FEATURES_GEOX, WELL_COL, and metadata
    (USE_DEPTH, DEPTH_BOUNDS, dataset_tag).
    """
    if dataset_tag not in DATASET_PATHS:
        raise KeyError(f"Unknown dataset_tag={dataset_tag!r}. Known: {list(DATASET_PATHS)}")

    train_data = np.load(_file(dataset_tag, "train{suffix}.npz"))
    val_data   = np.load(_file(dataset_tag, "val{suffix}.npz"))
    test_data  = np.load(_file(dataset_tag, "test{suffix}.npz"))
    df_test_g  = pd.read_parquet(_file(dataset_tag, "warm_df_test_g{suffix}.parquet"))
    tops_df    = pd.read_parquet(_file(dataset_tag, "warm_tops{suffix}.parquet"))
    with open(_file(dataset_tag, "warm_state{suffix}.json")) as f:
        ws = _json.load(f)

    out = {
        "X_train": train_data['X'],
        "y_train": train_data['y'].astype(np.int64),
        "X_val":   val_data['X'],
        "y_val":   val_data['y'].astype(np.int64),
        "X_test":  test_data['X'],
        "y_test":  test_data['y'].astype(np.int64),
        "df_test_g": df_test_g,
        "tops": tops_df,
        "test_wells": set(ws["test_wells"]),
        "well2id": {str(k): v for k, v in ws["well2id"].items()},
        "MARKERS": ws["MARKERS"],
        "FEATURES_GEOX": ws["FEATURES_GEOX"],
        "WELL_COL": ws["WELL_COL"],
        "USE_DEPTH": ws.get("USE_DEPTH", False),
        "DEPTH_BOUNDS": ws.get("DEPTH_BOUNDS"),
        "dataset_tag": dataset_tag,
    }

    # Install or remove the depth normalization patch to match this dataset
    if out["USE_DEPTH"] and 'Depth' in out["FEATURES_GEOX"]:
        _install_depth_patch(out["DEPTH_BOUNDS"])
    else:
        _uninstall_depth_patch()

    return out


def dataset_available(dataset_tag):
    """Check whether all artifacts for a dataset_tag exist on disk."""
    if dataset_tag not in DATASET_PATHS:
        return False
    needed = [
        _file(dataset_tag, "train{suffix}.npz"),
        _file(dataset_tag, "val{suffix}.npz"),
        _file(dataset_tag, "test{suffix}.npz"),
        _file(dataset_tag, "warm_df_test_g{suffix}.parquet"),
        _file(dataset_tag, "warm_tops{suffix}.parquet"),
        _file(dataset_tag, "warm_state{suffix}.json"),
    ]
    return all(Path(p).exists() for p in needed)


# Report which datasets are available on this machine
for tag in DATASET_PATHS:
    status = "OK" if dataset_available(tag) else "missing"
    print(f"  dataset_tag={tag!r:>10}  ->  {DATASET_PATHS[tag]['dir']}/  {status}")

In [ ]:
def build_test_state(ds):
    """Build the test-set evaluation state from a loaded dataset dict.

    Returns (test_log, test_tops_wide, pred_column).
    """
    test_log = ds["df_test_g"]
    test_tops_wide = (
        ds["tops"][ds["tops"][ds["WELL_COL"]].isin(ds["test_wells"])]
        .assign(wellName=lambda d: d[ds["WELL_COL"]].map(ds["well2id"]))
        .pivot_table(index="wellName", columns="marker_name", values="top_depth", aggfunc="first")
        .reindex(columns=ds["MARKERS"])
    )
    test_tops_wide.index.name = "wellName"
    pred_column = ['None'] + ds["MARKERS"]
    return test_log, test_tops_wide, pred_column


WSIZE = 201

In [ ]:
# --- Adapted Predicted_well_depth for N markers (original hardcodes 3) ---
# OOM-safe version: pre-groups by well, collects rows as dicts, monitors memory.

def Predicted_well_depth_npd(df_test_log, df_test_tops, dls, learn, pred_column, wsize, valid_dl, input_variable):
    well_list = list(df_test_tops.index)
    col = df_test_tops.reset_index().columns.tolist()

    # Pre-group by well so each get_markers call receives only its own subset
    # instead of scanning the full df_test_log frame each time.
    _well_groups = dict(list(df_test_log.groupby('wellName')))

    # Collect rows as dicts, build DataFrame once at the end (avoids repeated pd.concat).
    rows = []
    skipped = []

    for wi, well in enumerate(well_list):
        try:
            well_tops = df_test_tops.loc[well]
            existing_markers = well_tops.dropna().index.tolist()
            # Pass only this well's rows to get_markers (not the full frame).
            well_log = _well_groups.get(well, df_test_log.iloc[:0])
            pred_m, df_wm = get_markers(well_log, learn, dls, well, pred_column, wsize, valid_dl, input_variable, existing_markers)
            row = {col[0]: well}
            for j, depth in enumerate(pred_m):
                row[col[j + 1]] = depth
            rows.append(row)
            del df_wm, pred_m
        except Exception as e:
            skipped.append(well)
            continue

        # Progress + memory monitoring every 25 wells
        if (wi + 1) % 25 == 0 or (wi + 1) == len(well_list):
            gpu_mem = ""
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                a = torch.cuda.memory_allocated() / 1e9
                r = torch.cuda.memory_reserved() / 1e9
                gpu_mem = f" | GPU: {a:.2f}GB alloc, {r:.2f}GB reserved"
            try:
                import resource
                rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6
                cpu_mem = f" | RSS: {rss:.1f}GB"
            except Exception:
                cpu_mem = ""
            msg = "  Predicting: " + str(wi+1) + "/" + str(len(well_list)) + " wells" + gpu_mem + cpu_mem
            print(chr(13) + msg, end='')
            _sys.stdout.flush()
            _gc.collect()

    print()  # newline after progress
    del _well_groups
    _gc.collect()

    if skipped:
        print(f"  Skipped {len(skipped)}/{len(well_list)} wells (insufficient non-NaN log coverage for wsize={wsize})")

    df_tops_pred = pd.DataFrame(rows, columns=col).set_index(col[0])
    return df_tops_pred

WSIZE = 201


## Experiment plan & sharding (4 VMs, 4 shards)

Each arch-loss pair (from prior sweep results) is tested across all compatible feature subsets, with two lr strategies (fixed 1e-3 and lr_find). The plan is built from `ARCH_LOSS_PAIRS x SUBSETS_TO_RUN x LR_VARIANTS x DATASET_TAGS_TO_RUN`, filtering out incompatible subset-dataset combinations.

Every teammate runs the **same notebook** on their own VM; only `MY_SHARD` differs. The loop processes `RUN_PLAN[MY_SHARD::N_SHARDS]`, so there is no overlap between teammates.

### How to run

1. Set `MY_SHARD` to your assigned id (0, 1, 2, or 3) and leave `N_SHARDS = 4`.
2. Make sure both `depth/` and `no_depth/` dataset directories are present on disk.
3. Run all cells. Each run appends one row to `experiment_log__shard{MY_SHARD}.csv`.
4. After everyone finishes, copy CSVs to one machine and use the **Results viewer** cell.

### Registries

- **Subsets**: `FEATURE_SUBSETS_BY_NAME` - lists of feature *names*.
- **Architectures**: `ARCH_REGISTRY` - `(arch_class, arch_kwargs)` keyed by label.
- **Arch-loss pairs**: `ARCH_LOSS_PAIRS` - each arch with its best loss from the sweep.
- **Common recipe**: `COMMON_RECIPE` - shared training hyperparameters for all runs.
- **LR variants**: `LR_VARIANTS` - `[False, True]` to test fixed lr vs lr_find.

### Ranking

Results are ranked by **recall** (primary), then MAE, then max_error. Accuracy is logged but is not the selection metric.


In [ ]:
# ---- Shard configuration ----
# Set MY_SHARD on each VM. N_SHARDS must match the number of teammates.
# Every VM must build the SAME global RUN_PLAN so that shards partition cleanly.
MY_SHARD = 0          # 0, 1, 2, or 3
N_SHARDS = 4

RECALL_TOLERANCE = 50
WSIZE            = 201

# ---- Feature subset registry (by NAME) ----
# Use the feature name strings as they appear in warm_state*.json["FEATURES_GEOX"].
# Some subsets reference "Depth" -- only valid against the "depth" dataset,
# filtered out automatically by the plan builder.
FEATURE_SUBSETS_BY_NAME = {
    "all_logs_latlon":     ["GR", "RHOB", "DTC", "DEEPRES", "NEUT", "Latitude", "Longitude"],
    "all_logs_latlon_d":   ["GR", "RHOB", "DTC", "DEEPRES", "NEUT", "Latitude", "Longitude", "Depth"],
    "GR_only":             ["GR"],
    "GR_latlon":           ["GR", "Latitude", "Longitude"],
    "GR_latlon_depth":     ["GR", "Latitude", "Longitude", "Depth"],
    "high_coverage":       ["GR", "DEEPRES", "Latitude", "Longitude"],
    "high_coverage_depth": ["GR", "DEEPRES", "Latitude", "Longitude", "Depth"],
    "no_latlon":           ["GR", "RHOB", "DTC", "DEEPRES", "NEUT"],
}

# Static dataset-agnostic compatibility map (so every VM builds identical plans).
DATASET_FEATURES = {
    "nodepth": {"GR", "RHOB", "DTC", "DEEPRES", "NEUT", "Latitude", "Longitude"},
    "depth":   {"GR", "RHOB", "DTC", "DEEPRES", "NEUT", "Latitude", "Longitude", "Depth"},
}

def resolve_subset(subset_name, features_geox):
    """Resolve a named subset to channel indices into FEATURES_GEOX.
    Returns (indices, names) or raises ValueError if any feature is missing."""
    names = FEATURE_SUBSETS_BY_NAME[subset_name]
    missing = [n for n in names if n not in features_geox]
    if missing:
        raise ValueError(
            f"Subset {subset_name!r} references features {missing} not in FEATURES_GEOX={features_geox}"
        )
    indices = [features_geox.index(n) for n in names]
    return indices, names

# ---- Architecture registry ----
ARCH_REGISTRY = {
    "LSTM_2dCNN_default":   (LSTM_2dCNN, {}),
    "LSTM_XCM_default":     (LSTM_XCM,   {}),
    "LSTM_FCN_default":     (LSTM_FCN,   {}),
}

# ---- Arch-loss pairs (from prior loss sweep results) ----
# LSTM_2dCNN -> label_smooth (best overall balance)
# LSTM_FCN   -> label_smooth (best recall, MAE, max_error)
# LSTM_XCM   -> ce (best recall) + focal (best MAE/max_error) -- tradeoff still open
ARCH_LOSS_PAIRS = [
    ("LSTM_2dCNN_default", "label_smooth"),
    ("LSTM_FCN_default",   "label_smooth"),
    ("LSTM_XCM_default",   "ce"),
    ("LSTM_XCM_default",   "focal"),
]

# ---- Common training recipe (shared by all runs) ----
# Loss type and lr_find are set per-run from ARCH_LOSS_PAIRS and LR_VARIANTS.
COMMON_RECIPE = {
    "fixed_lr":             1e-3,
    "monitor":              "valid_loss",
    "early_stop_patience":  6,
    "early_stop_min_delta": 0.0,
    "n_epochs":             25,
    "batch_size":           25,
}

# ---- LR variants ----
# False = fixed lr (matches original sweep conditions)
# True  = lr_find (discovers lr per model)
LR_VARIANTS = [False, True]

# ---- Run plan builder ----
DATASET_TAGS_TO_RUN = ["nodepth", "depth"]
SUBSETS_TO_RUN = [
    "all_logs_latlon",
    "all_logs_latlon_d",
    "high_coverage",
    "high_coverage_depth",
    "GR_latlon",
    "GR_latlon_depth",
    "no_latlon",
    "GR_only",
]

def _subset_compatible_static(subset_name, dataset_tag):
    expected = DATASET_FEATURES.get(dataset_tag, set())
    return all(n in expected for n in FEATURE_SUBSETS_BY_NAME[subset_name])

def build_run_plan():
    """Build the full run plan: arch-loss pairs x subsets x lr variants x datasets."""
    plan = []
    for dataset_tag in DATASET_TAGS_TO_RUN:
        for subset_name in SUBSETS_TO_RUN:
            if not _subset_compatible_static(subset_name, dataset_tag):
                continue
            for arch_label, loss_type in ARCH_LOSS_PAIRS:
                for use_lr_find in LR_VARIANTS:
                    lr_tag = "lr_find" if use_lr_find else "lr_fixed"
                    run_name = f"{arch_label}__{loss_type}__{subset_name}__{lr_tag}"
                    plan.append({
                        "dataset_tag":  dataset_tag,
                        "subset_name":  subset_name,
                        "arch_label":   arch_label,
                        "loss_type":    loss_type,
                        "use_lr_find":  use_lr_find,
                        "run_name":     run_name,
                    })
    return plan

RUN_PLAN = build_run_plan()
MY_RUN_PLAN = RUN_PLAN[MY_SHARD::N_SHARDS]

# Pre-flight report
runnable_count = sum(dataset_available(c["dataset_tag"]) for c in MY_RUN_PLAN)
missing_datasets = sorted({c["dataset_tag"] for c in MY_RUN_PLAN if not dataset_available(c["dataset_tag"])})

print(f"Global RUN_PLAN size:    {len(RUN_PLAN)} runs")
print(f"This shard ({MY_SHARD}/{N_SHARDS}):       {len(MY_RUN_PLAN)} runs")
print(f"Runnable on this VM:     {runnable_count} runs")
if missing_datasets:
    print(f"\n  WARNING: dataset_tag(s) missing on this VM: {missing_datasets}")
    print(f"  -> {len(MY_RUN_PLAN) - runnable_count} runs will be SKIPPED.")
print(f"\nFirst 5 runs in this shard:")
for i, combo in enumerate(MY_RUN_PLAN[:5]):
    avail = "OK" if dataset_available(combo["dataset_tag"]) else "MISSING"
    print(f"  [{i}] {avail:7s} {combo['run_name']}")


In [ ]:
def compute_class_weights(y_train):
    """Compute inverse-frequency class weights from training labels.

    Returns a torch float tensor of shape (n_classes,) ready to pass as
    the `weight` argument to nn.CrossEntropyLoss / CrossEntropyLossFlat.
    """
    classes, counts = np.unique(y_train, return_counts=True)
    weights = len(y_train) / (len(classes) * counts)
    # Ensure the tensor covers class indices 0..max contiguously
    w = np.zeros(int(classes.max()) + 1, dtype=np.float32)
    for c, wt in zip(classes, weights):
        w[int(c)] = wt
    return torch.tensor(w, dtype=torch.float32)


def build_dataloader(ds, subset_idx, batch_size):
    """Slice the chosen feature channels and build a tsai TSDataLoaders.

    Returns (dls, valid_dl, X_train_sub_shape).
    """
    X_train_sub = ds["X_train"][:, subset_idx, :]
    X_val_sub   = ds["X_val"][:,   subset_idx, :]
    X, y, splits = combine_split_data([X_train_sub, X_val_sub], [ds["y_train"], ds["y_val"]])
    tfms  = [None, [Categorize()]]
    dsets = TSDatasets(X, y, tfms=tfms, splits=splits, inplace=True)
    dls = TSDataLoaders.from_dsets(
        dsets.train, dsets.valid,
        bs=batch_size, batch_tfms=[], num_workers=0,
    )
    return dls, dls.valid, X_train_sub.shape

# ---- Loss function builder (matches combined_data_split.ipynb sweep) ----
ENS_BETA = 0.999

def compute_ens_weights(y_train):
    """Effective Number of Samples weights (Cui et al. 2019)."""
    classes, counts = np.unique(y_train, return_counts=True)
    ens = (1 - ENS_BETA) / (1 - ENS_BETA ** counts)
    ens = ens / ens.min()
    w = np.zeros(int(classes.max()) + 1, dtype=np.float32)
    for c, wt in zip(classes, ens):
        w[int(c)] = wt
    return torch.tensor(w, dtype=torch.float32)

def compute_sqrt_weights(y_train):
    """Sqrt-dampened inverse-frequency weights (gentler than full CWL)."""
    classes, counts = np.unique(y_train, return_counts=True)
    w_full = len(y_train) / (len(classes) * counts)
    w_sqrt = np.sqrt(w_full)
    w_sqrt = w_sqrt / w_sqrt.min()
    w = np.zeros(int(classes.max()) + 1, dtype=np.float32)
    for c, wt in zip(classes, w_sqrt):
        w[int(c)] = wt
    return torch.tensor(w, dtype=torch.float32)

def build_loss(loss_type, class_weights, ens_weights, sqrt_weights, device):
    """Build loss function from loss_type string."""
    if loss_type == "ce":           return CrossEntropyLossFlat()
    elif loss_type == "cwl":        return CrossEntropyLossFlat(weight=class_weights.to(device))
    elif loss_type == "cwl_sqrt":   return CrossEntropyLossFlat(weight=sqrt_weights.to(device))
    elif loss_type == "ens":        return CrossEntropyLossFlat(weight=ens_weights.to(device))
    elif loss_type == "focal":      return FocalLossFlat(gamma=2.0)
    elif loss_type == "focal_cwl":  return FocalLossFlat(gamma=2.0, weight=class_weights.to(device))
    elif loss_type == "focal_ens":  return FocalLossFlat(gamma=2.0, weight=ens_weights.to(device))
    elif loss_type == "label_smooth": return LabelSmoothingCrossEntropyFlat(eps=0.1)
    else: raise ValueError(f"Unknown loss_type={loss_type!r}")


In [ ]:
# --- Training & evaluation loop over MY_RUN_PLAN ---------------------------
LOG_PATH = Path(f"experiment_log__shard{MY_SHARD}.csv")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# Cache loaded datasets so each dataset_tag's .npz is read at most once.
_dataset_cache = {}
_test_state_cache = {}
_weight_cache = {}
def _get_dataset(tag):
    if tag not in _dataset_cache:
        _dataset_cache[tag] = load_dataset(tag)
        _test_state_cache[tag] = build_test_state(_dataset_cache[tag])
        y = _dataset_cache[tag]["y_train"]
        _weight_cache[tag] = {
            "cwl":  compute_class_weights(y),
            "ens":  compute_ens_weights(y),
            "sqrt": compute_sqrt_weights(y),
        }
    return _dataset_cache[tag], _test_state_cache[tag], _weight_cache[tag]

result_recall = pd.DataFrame(columns=['run_name', 'arch', 'loss', 'subset', 'lr_strategy', 'dataset', 'recall', 'mae', 'max_error', 'eval_time'])
results       = pd.DataFrame(columns=['run_name', 'arch', 'loss', 'subset', 'lr_strategy', 'dataset', 'train_loss', 'valid_loss', 'accuracy', 'train_time'])

for combo_idx, combo in enumerate(MY_RUN_PLAN):
    dataset_tag  = combo["dataset_tag"]
    subset_name  = combo["subset_name"]
    arch_label   = combo["arch_label"]
    loss_type    = combo["loss_type"]
    use_lr_find  = combo["use_lr_find"]
    run_name     = combo["run_name"]
    lr_tag       = "lr_find" if use_lr_find else "lr_fixed"

    print(f"\n[{combo_idx+1}/{len(MY_RUN_PLAN)}] {run_name}")

    # Reset seed before each run for reproducibility
    random_seed(0, use_cuda=True)

    # Load dataset (cached)
    try:
        ds, (test_log, test_tops_wide, pred_column), weights = _get_dataset(dataset_tag)
    except FileNotFoundError as e:
        print(f"  SKIP: dataset {dataset_tag!r} missing ({e.filename})")
        continue

    try:
        subset_idx, subset_names = resolve_subset(subset_name, ds["FEATURES_GEOX"])
    except ValueError as e:
        print(f"  SKIP: {e}")
        continue

    arch_class, arch_kwargs = ARCH_REGISTRY[arch_label]
    R = COMMON_RECIPE

    # Build dataloader
    dls, valid_dl, train_shape = build_dataloader(ds, subset_idx, R["batch_size"])
    print(f"  shape={train_shape}, loss={loss_type}, lr={lr_tag}")

    # Build model + loss
    model = build_model(arch_class, dls=dls, **arch_kwargs)
    device = next(model.parameters()).device
    loss_func = build_loss(loss_type, weights["cwl"], weights["ens"], weights["sqrt"], device)
    learn = Learner(dls, model, loss_func=loss_func, metrics=accuracy)

    # Learning rate
    if use_lr_find:
        try:
            lr_result = learn.lr_find(show_plot=False)
            lr = lr_result.valley if hasattr(lr_result, 'valley') else float(lr_result)
        except Exception:
            lr = R["fixed_lr"]
    else:
        lr = R["fixed_lr"]

    cbs = [
        SaveModelCallback(monitor=R["monitor"]),
        EarlyStoppingCallback(
            monitor='valid_loss',
            min_delta=R["early_stop_min_delta"],
            patience=R["early_stop_patience"],
        ),
    ]
    start = time.time()
    learn.fit_one_cycle(R["n_epochs"], lr, cbs=cbs)
    train_elapsed = time.time() - start
    vals = learn.recorder.values[-1]
    best_epoch = len(learn.recorder.values)

    # Save model checkpoint
    model_path = MODEL_DIR / f"{run_name}.pth"
    torch.save(learn.model.state_dict(), model_path)

    # Evaluate on test set
    eval_start = time.time()
    df_tops_pred = Predicted_well_depth_npd(
        df_test_log=test_log, df_test_tops=test_tops_wide,
        dls=dls, learn=learn, pred_column=pred_column,
        wsize=WSIZE, valid_dl=valid_dl, input_variable=subset_names,
    )
    recall, mae, df_result = recall_tops(test_tops_wide, df_tops_pred, tolerance=RECALL_TOLERANCE)
    eval_elapsed = time.time() - eval_start

    ae_cols = [c for c in df_result.columns if c.endswith('_ae')]
    max_error_arr = df_result[ae_cols].values.astype(float)
    max_error = float(np.nanmax(max_error_arr)) if not np.all(np.isnan(max_error_arr)) else np.nan

    # Update in-memory display tables
    result_recall.loc[combo_idx] = [
        run_name, arch_label, loss_type, subset_name, lr_tag, dataset_tag,
        recall, mae, max_error, int(eval_elapsed),
    ]
    results.loc[combo_idx] = [
        run_name, arch_label, loss_type, subset_name, lr_tag, dataset_tag,
        vals[0], vals[1], vals[2], int(train_elapsed),
    ]
    result_recall.sort_values(by=['recall', 'max_error'], ascending=[False, True], kind='stable', ignore_index=True, inplace=True)
    results.sort_values(by='accuracy', ascending=False, kind='stable', ignore_index=True, inplace=True)

    # Append to persistent CSV log
    log_row = {
        "run_id":              str(uuid.uuid4()),
        "run_name":            run_name,
        "timestamp":           datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "machine":             MACHINE,
        "git_sha":             GIT_SHA,
        "shard":               MY_SHARD,
        "n_shards":            N_SHARDS,
        "dataset_tag":         dataset_tag,
        "use_depth":           bool(ds["USE_DEPTH"]),
        "subset_name":         subset_name,
        "feature_names":       ",".join(subset_names),
        "n_channels":          len(subset_idx),
        "arch_label":          arch_label,
        "arch_class":          arch_class.__name__,
        "arch_kwargs":         str(arch_kwargs),
        "loss_type":           loss_type,
        "use_lr_find":         use_lr_find,
        "lr":                  float(lr),
        "monitor":             R["monitor"],
        "patience":            R["early_stop_patience"],
        "min_delta":           R["early_stop_min_delta"],
        "n_epochs":            R["n_epochs"],
        "batch_size":          R["batch_size"],
        "best_epoch":          best_epoch,
        "total_params":        int(count_parameters(model)),
        "train_loss":          float(vals[0]),
        "valid_loss":          float(vals[1]),
        "accuracy":            float(vals[2]),
        "recall":              float(recall) if recall is not None else np.nan,
        "mae":                 float(mae) if mae is not None else np.nan,
        "max_error":           max_error,
        "tolerance":           RECALL_TOLERANCE,
        "train_time_s":        int(train_elapsed),
        "eval_time_s":         int(eval_elapsed),
        "model_path":          str(model_path),
    }
    log_df = pd.DataFrame([log_row])
    if LOG_PATH.exists():
        log_df.to_csv(LOG_PATH, mode="a", header=False, index=False)
    else:
        log_df.to_csv(LOG_PATH, mode="w", header=True, index=False)

    # GPU cleanup to prevent OOM on VMs with limited RAM
    learn.model.cpu()
    del learn, model, loss_func
    del df_tops_pred, df_result
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    _gc.collect(); _gc.collect()
    try:
        import ctypes
        ctypes.CDLL('libc.so.6').malloc_trim(0)  # Linux: release freed memory back to OS
    except Exception:
        pass

    clear_output()
    display(result_recall)
    display(results)

print(f"\nDone. Logged {len(MY_RUN_PLAN)} runs to {LOG_PATH.resolve()}")


## Results viewer

Concatenates every `experiment_log__shard*.csv` present in the working directory. After all teammates finish, copy their CSVs into one folder and run this cell to see the merged leaderboard. **Primary ranking: recall** (higher is better), then MAE (lower), then max_error (lower).


In [ ]:
# Merge all per-shard CSV logs and display the leaderboard
shard_files = sorted(Path(".").glob("experiment_log__shard*.csv"))
if not shard_files:
    print("No experiment_log__shard*.csv files found yet. Run the training loop first.")
else:
    print(f"Found {len(shard_files)} shard files: {[p.name for p in shard_files]}")
    log_all = pd.concat([pd.read_csv(p) for p in shard_files], ignore_index=True)
    print(f"Total rows: {len(log_all)}")

    display_cols = ["run_name", "dataset_tag", "subset_name", "arch_label", "loss_type",
                    "use_lr_find", "recall", "accuracy", "mae", "max_error", "machine", "shard"]
    # Only show columns that exist in the CSV
    display_cols = [c for c in display_cols if c in log_all.columns]

    print()
    print("Top 20 by recall:")
    display(
        log_all.sort_values(["recall", "max_error"], ascending=[False, True])
               .head(20)[display_cols]
    )
    print()
    print("Top 20 by accuracy:")
    display(
        log_all.sort_values("accuracy", ascending=False)
               .head(20)[display_cols]
    )

    # Best subset per arch-loss pair
    print()
    print("Best run per arch-loss pair (by recall):")
    if "arch_label" in log_all.columns and "loss_type" in log_all.columns:
        best_per_pair = (
            log_all.sort_values(["recall", "mae", "max_error"], ascending=[False, True, True])
                   .groupby(["arch_label", "loss_type"])
                   .first()
                   .reset_index()
        )
        display(best_per_pair[display_cols] if all(c in best_per_pair.columns for c in display_cols) else best_per_pair)


In [ ]:
# --- Re-run the best model from the experiment log ---
# After reviewing results in the viewer above, set BEST_* to the winning combo.
# This cell trains that single model and keeps `learn_best` alive for further analysis
# (confusion matrix, per-well inspection, model export, etc.)

BEST_DATASET    = "nodepth"              # dataset_tag
BEST_SUBSET     = "all_logs_latlon"      # subset name
BEST_ARCH       = "LSTM_FCN_default"     # arch label from ARCH_REGISTRY
BEST_LOSS       = "label_smooth"         # loss type
BEST_LR_FIND    = False                  # True = lr_find, False = fixed 1e-3

# ---- Load & build ----
ds_best = load_dataset(BEST_DATASET)
test_log_best, test_tops_best, pred_column_best = build_test_state(ds_best)
subset_idx_best, subset_names_best = resolve_subset(BEST_SUBSET, ds_best["FEATURES_GEOX"])
R = COMMON_RECIPE

dls_best, valid_dl_best, _ = build_dataloader(ds_best, subset_idx_best, R["batch_size"])

model_best = build_model(*ARCH_REGISTRY[BEST_ARCH], dls=dls_best)
device_best = next(model_best.parameters()).device

y_best = ds_best["y_train"]
wt_best = {
    "cwl":  compute_class_weights(y_best),
    "ens":  compute_ens_weights(y_best),
    "sqrt": compute_sqrt_weights(y_best),
}
loss_best = build_loss(BEST_LOSS, wt_best["cwl"], wt_best["ens"], wt_best["sqrt"], device_best)
learn_best = Learner(dls_best, model_best, loss_func=loss_best, metrics=accuracy)

# LR
if BEST_LR_FIND:
    try:
        lr_res = learn_best.lr_find(show_plot=False)
        lr_best = lr_res.valley if hasattr(lr_res, 'valley') else float(lr_res)
    except Exception:
        lr_best = R["fixed_lr"]
else:
    lr_best = R["fixed_lr"]

cbs_best = [
    SaveModelCallback(monitor=R["monitor"]),
    EarlyStoppingCallback(
        monitor='valid_loss',
        min_delta=R["early_stop_min_delta"],
        patience=R["early_stop_patience"],
    ),
]

random_seed(0, use_cuda=True)
start_best = time.time()
learn_best.fit_one_cycle(R["n_epochs"], lr_best, cbs=cbs_best)
elapsed_best = time.time() - start_best
vals_best = learn_best.recorder.values[-1]
print(f"\nTrain loss: {vals_best[0]:.4f} | Valid loss: {vals_best[1]:.4f} | Accuracy: {vals_best[2]:.4f} | Time: {int(elapsed_best)}s")

# Save best model
best_model_path = Path("models") / f"best__{BEST_ARCH}__{BEST_LOSS}__{BEST_SUBSET}.pth"
best_model_path.parent.mkdir(exist_ok=True)
torch.save(learn_best.model.state_dict(), best_model_path)
print(f"Model saved to {best_model_path}")

# Evaluate recall
df_tops_pred_best = Predicted_well_depth_npd(
    df_test_log=test_log_best, df_test_tops=test_tops_best,
    dls=dls_best, learn=learn_best, pred_column=pred_column_best,
    wsize=WSIZE, valid_dl=valid_dl_best, input_variable=subset_names_best,
)
recall_best, mae_best, df_result_best = recall_tops(test_tops_best, df_tops_pred_best, tolerance=RECALL_TOLERANCE)
print(f"Recall: {recall_best:.3f} | MAE: {mae_best:.1f}")
print(f"\nlearn_best is ready for show_confusion_matrix_heatmap(learn_best, dls_best, pred_column_best)")


In [ ]:
# --- Confusion matrix heatmap ---
# Call after the best-model cell above (uses learn_best, dls_best, pred_column_best).
# Or pass any (learn, dls, pred_labels) from any training run.

def show_confusion_matrix_heatmap(learn, dls, pred_labels, title="Confusion Matrix", figsize=(10, 8)):
    """Display a confusion matrix as a seaborn heatmap with per-class accuracy."""
    val_preds, val_targs = learn.get_preds(dl=dls.valid)
    val_pred_classes = val_preds.argmax(dim=1).numpy()
    val_true_classes = val_targs.numpy()

    cm = confusion_matrix(val_true_classes, val_pred_classes, labels=list(range(len(pred_labels))))

    # Normalized version (row-wise: each row sums to 1)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

    fig, axes = plt.subplots(1, 2, figsize=(figsize[0] * 2, figsize[1]))

    # Left: raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=pred_labels, yticklabels=pred_labels, ax=axes[0])
    axes[0].set_title(f"{title} (counts)")
    axes[0].set_ylabel("True class")
    axes[0].set_xlabel("Predicted class")

    # Right: normalized (percentages)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=pred_labels, yticklabels=pred_labels, ax=axes[1])
    axes[1].set_title(f"{title} (normalized)")
    axes[1].set_ylabel("True class")
    axes[1].set_xlabel("Predicted class")

    plt.tight_layout()
    plt.show()

    # Per-class accuracy table
    per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    print("Per-class accuracy:")
    for lbl, acc_val in zip(pred_labels, per_class_acc):
        print(f"  {lbl:15s}: {acc_val:.3f}")
    print(f"  {'OVERALL':15s}: {cm.diagonal().sum() / cm.sum():.3f}")

# Run it on the best model
show_confusion_matrix_heatmap(learn_best, dls_best, pred_column_best)
